In [ ]:
# process UPLC data- exported as csv

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

# ============================================================================
# CONFIGURATION
# ============================================================================

# Place your data in folders relative to this script
INPUT_DIRECTORY = "./data/input"
OUTPUT_DIRECTORY = "./results"

# TWO BLANK FILES - Define paths for both wavelengths
BLANK_FILE_A_PATH = "./data/blanks/BLANK1_214nm.CSV"  # Blank for 214 nm (A files)
BLANK_FILE_B_PATH = "./data/blanks/BLANK1_280nm.CSV"  # Blank for 280 nm (B files)
# Plot parameters
PLOT_DPI = 300
FIGURE_SIZE_NARROW = (14, 8)
FIGURE_SIZE_EXTRA_NARROW = (12, 8)

# Styling
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['lines.linewidth'] = 1.5

# ============================================================================
# FUNCTION 1: Read CSV File with UTF-16 Encoding
# ============================================================================

def read_csv_file(filepath):
    """Read CSV file with UTF-16 encoding and tab/space delimiter."""
    try:
        # Try UTF-16 first (Agilent format)
        data = pd.read_csv(
            filepath, 
            header=None, 
            encoding='utf-16',
            delimiter=r'\s+',  # Any whitespace (space or tab)
            engine='python'
        )
        
        if data.shape[1] < 2:
            print(f"  ✗ File has less than 2 columns")
            return None, None
        
        time = data.iloc[:, 0].values.astype(float)
        absorbance = data.iloc[:, 1].values.astype(float)
        
        return time, absorbance
    
    except UnicodeDecodeError:
        # Fallback to UTF-8 if UTF-16 fails
        try:
            data = pd.read_csv(
                filepath,
                header=None,
                encoding='utf-8',
                delimiter=r'\s+',
                engine='python'
            )
            
            if data.shape[1] < 2:
                print(f"  ✗ File has less than 2 columns")
                return None, None
            
            time = data.iloc[:, 0].values.astype(float)
            absorbance = data.iloc[:, 1].values.astype(float)
            
            return time, absorbance
        except Exception as e:
            print(f"  ✗ Error reading {filepath.name}: {e}")
            return None, None
    
    except Exception as e:
        print(f"  ✗ Error reading {filepath.name}: {e}")
        return None, None

# ============================================================================
# FUNCTION 2: Load Blank Files
# ============================================================================

def load_blank_files(blank_a_path, blank_b_path):
    """Load both blank CSV files."""
    
    blank_a_time = None
    blank_a_absorbance = None
    blank_b_time = None
    blank_b_absorbance = None
    
    print(f"Loading blank files...\n")
    
    # Load blank A (214 nm)
    blank_a_path = Path(blank_a_path)
    if blank_a_path.exists():
        print(f"✓ Found blank A file (214 nm): {blank_a_path.name}")
        blank_a_time, blank_a_absorbance = read_csv_file(blank_a_path)
        if blank_a_time is not None:
            print(f"  Data points: {len(blank_a_time)}\n")
    else:
        print(f"✗ Blank A file (214 nm) not found: {blank_a_path}\n")
    
    # Load blank B (280 nm)
    blank_b_path = Path(blank_b_path)
    if blank_b_path.exists():
        print(f"✓ Found blank B file (280 nm): {blank_b_path.name}")
        blank_b_time, blank_b_absorbance = read_csv_file(blank_b_path)
        if blank_b_time is not None:
            print(f"  Data points: {len(blank_b_time)}\n")
    else:
        print(f"✗ Blank B file (280 nm) not found: {blank_b_path}\n")
    
    return (blank_a_time, blank_a_absorbance), (blank_b_time, blank_b_absorbance)

# ============================================================================
# FUNCTION 3: Apply Background Correction
# ============================================================================

def apply_background_correction(absorbance, blank_time, blank_absorbance, sample_time):
    """Subtract blank absorbance from sample absorbance."""
    if blank_absorbance is None:
        print("  ⚠ No blank available, using uncorrected data")
        return absorbance
    
    # Interpolate blank to match sample time points
    if len(blank_time) != len(sample_time) or not np.allclose(blank_time, sample_time):
        blank_absorbance_interp = np.interp(sample_time, blank_time, blank_absorbance)
    else:
        blank_absorbance_interp = blank_absorbance
    
    # Subtract blank
    corrected = absorbance - blank_absorbance_interp
    
    # Ensure no negative values
    corrected = np.maximum(corrected, 0)
    
    return corrected

# ============================================================================
# FUNCTION 4: Generate Full Plot (Black line, no fill)
# ============================================================================

def generate_plot(filepath, time, absorbance, wavelength):
    """Generate and save full range plot (black line, no fill)."""
    
    output_filename = f"{filepath.stem}.png"
    output_path = Path(OUTPUT_DIRECTORY) / output_filename
    
    # Create figure
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_NARROW)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    # Plot BLACK LINE ONLY (no fill)
    ax.plot(time, absorbance, linewidth=2.5, color='black')
    
    # Large fonts
    ax.set_xlabel('Time (min)', fontsize=26, fontweight='bold', labelpad=15)
    ax.set_ylabel(f'Absorbance {wavelength} nm', fontsize=26, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='major', width=2.5, length=10)
    
    # Professional styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)
    
    # No grid
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved: {output_filename}")
    return output_path

# ============================================================================
# FUNCTION 5: Generate SHORT Plot (3-20 min, narrower, black line)
# ============================================================================

def generate_plot_short(filepath, time, absorbance, wavelength):
    """Generate and save SHORT range plot (3-20 min, narrower, black line, no fill)."""
    
    output_filename = f"{filepath.stem}_SHORT.png"
    output_path = Path(OUTPUT_DIRECTORY) / output_filename
    
    # Filter data for 3-20 min range
    mask = (time >= 3.0) & (time <= 15.0)
    filtered_time = time[mask]
    filtered_absorbance = absorbance[mask]
    
    if len(filtered_time) == 0:
        print(f"  ⚠ No data in 3-20 min range for SHORT plot")
        return None
    
    # Create figure (narrower)
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_EXTRA_NARROW)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    # Plot BLACK LINE ONLY (no fill)
    ax.plot(filtered_time, filtered_absorbance, linewidth=2.5, color='black')
    
    # Large fonts
    ax.set_xlabel('Time (min)', fontsize=26, fontweight='bold', labelpad=15)
    ax.set_ylabel(f'Absorbance {wavelength} nm', fontsize=26, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='major', width=2.5, length=10)
    
    # Professional styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)
    
    # Set x-axis limits
    ax.set_xlim(3.0, 15.0)
    
    # No grid
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved: {output_filename}")
    return output_path

# ============================================================================
# FUNCTION 6: Detect File Type
# ============================================================================

def get_file_type(filename):
    """Determine if file is A (214 nm) or B (280 nm)."""
    filename_upper = filename.upper()
    
    if '_DAD1A' in filename_upper or filename_upper.endswith('A.CSV'):
        return 'A', '214'
    elif '_DAD1B' in filename_upper or filename_upper.endswith('B.CSV'):
        return 'B', '280'
    else:
        return None, None

# ============================================================================
# FUNCTION 7: Process All Files
# ============================================================================

def process_all_csv_files():
    """Process all CSV files in the directory."""
    
    input_path = Path(INPUT_DIRECTORY)
    output_path = Path(OUTPUT_DIRECTORY)
    
    if not input_path.exists():
        print(f"✗ Directory not found: {INPUT_DIRECTORY}")
        return
    
    output_path.mkdir(exist_ok=True)
    
    # Load blank files
    (blank_a_time, blank_a_absorbance), (blank_b_time, blank_b_absorbance) = load_blank_files(
        BLANK_FILE_A_PATH, 
        BLANK_FILE_B_PATH
    )
    
    # Find all CSV files
    all_csv_files = sorted(list(input_path.glob('*.CSV'))) + sorted(list(input_path.glob('*.csv')))
    all_csv_files = list(dict.fromkeys(all_csv_files))  # Remove duplicates
    
    if not all_csv_files:
        print(f"✗ No CSV files found in: {INPUT_DIRECTORY}")
        return
    
    print(f"{'='*70}")
    print(f"Processing {len(all_csv_files)} CSV files")
    print(f"{'='*70}\n")
    
    success_a = 0
    success_b = 0
    failed = 0
    skipped = 0
    
    for filepath in all_csv_files:
        try:
            # Skip blank files
            if 'BLANK' in filepath.name.upper():
                print(f"⊘ Skipping blank file: {filepath.name}")
                skipped += 1
                continue
            
            # Determine file type
            file_letter, wavelength = get_file_type(filepath.name)
            
            if file_letter is None:
                print(f"⚠ Skipping {filepath.name} (cannot determine file type)")
                skipped += 1
                continue
            
            file_type = f"{file_letter} ({wavelength} nm)"
            print(f"Processing: {filepath.name} ({file_type})")
            
            # Read file
            time, absorbance = read_csv_file(filepath)
            
            if time is None or absorbance is None:
                failed += 1
                continue
            
            # Select appropriate blank
            if file_letter == 'A':
                blank_time = blank_a_time
                blank_absorbance = blank_a_absorbance
                print(f"  ✓ Using blank A (214 nm) for background subtraction")
            else:
                blank_time = blank_b_time
                blank_absorbance = blank_b_absorbance
                print(f"  ✓ Using blank B (280 nm) for background subtraction")
            
            # Apply background correction
            if blank_absorbance is not None:
                absorbance_corrected = apply_background_correction(
                    absorbance, blank_time, blank_absorbance, time
                )
                print(f"  Background corrected")
            else:
                absorbance_corrected = absorbance
                print(f"  No background correction applied")
            
            # Generate full range plot
            generate_plot(filepath, time, absorbance_corrected, wavelength)
            
            # Generate SHORT range plot (3-20 min)
            generate_plot_short(filepath, time, absorbance_corrected, wavelength)
            
            if file_letter == 'A':
                success_a += 1
            else:
                success_b += 1
        
        except Exception as e:
            print(f"✗ Error processing {filepath.name}: {e}")
            failed += 1
    
    print(f"\n{'='*70}")
    print(f"Complete!")
    print(f"  ✓ Processed A files (214 nm): {success_a}")
    print(f"  ✓ Processed B files (280 nm): {success_b}")
    if skipped > 0:
        print(f"  ⊘ Skipped: {skipped}")
    if failed > 0:
        print(f"  ✗ Failed: {failed}")
    print(f"{'='*70}\n")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    process_all_csv_files()

In [ ]:
#UPLC-data-with integrals

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from scipy.integrate import trapezoid

# ============================================================================
# CONFIGURATION
# ============================================================================


# Place your data in folders relative to this script
INPUT_DIRECTORY = "./data/input"
OUTPUT_DIRECTORY = "./results"

# TWO BLANK FILES - Define paths for both wavelengths
BLANK_FILE_A_PATH = "./data/blanks/BLANK1_214nm.CSV"  # Blank for 214 nm (A files)
BLANK_FILE_B_PATH = "./data/blanks/BLANK1_280nm.CSV"  # Blank for 280 nm (B files)

# Integration settings
ENABLE_PEAK_INTEGRATION = True
PEAK_HEIGHT_THRESHOLD = 0.10
PEAK_DISTANCE = 5
DECONVOLUTION_THRESHOLD = 0.50  # Only deconvolute if overlapping peaks > 50% of TIC (major peak intensity)

# Plot parameters
PLOT_DPI = 300
FIGURE_SIZE_NARROW = (14, 8)
FIGURE_SIZE_EXTRA_NARROW = (12, 8)

# Styling
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['lines.linewidth'] = 1.5

# Colors for peaks
COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE', '#95E1D3']

# ============================================================================
# FUNCTION 1: Read CSV File with UTF-16 Encoding
# ============================================================================

def read_csv_file(filepath):
    """Read CSV file with UTF-16 encoding and tab/space delimiter."""
    try:
        data = pd.read_csv(
            filepath, 
            header=None, 
            encoding='utf-16',
            delimiter=r'\s+',
            engine='python'
        )
        
        if data.shape[1] < 2:
            print(f"  ✗ File has less than 2 columns")
            return None, None
        
        time = data.iloc[:, 0].values.astype(float)
        absorbance = data.iloc[:, 1].values.astype(float)
        
        return time, absorbance
    
    except UnicodeDecodeError:
        try:
            data = pd.read_csv(
                filepath,
                header=None,
                encoding='utf-8',
                delimiter=r'\s+',
                engine='python'
            )
            
            if data.shape[1] < 2:
                print(f"  ✗ File has less than 2 columns")
                return None, None
            
            time = data.iloc[:, 0].values.astype(float)
            absorbance = data.iloc[:, 1].values.astype(float)
            
            return time, absorbance
        except Exception as e:
            print(f"  ✗ Error reading {filepath.name}: {e}")
            return None, None
    
    except Exception as e:
        print(f"  ✗ Error reading {filepath.name}: {e}")
        return None, None

# ============================================================================
# FUNCTION 2: Load Blank Files
# ============================================================================

def load_blank_files(blank_a_path, blank_b_path):
    """Load both blank CSV files."""
    
    blank_a_time = None
    blank_a_absorbance = None
    blank_b_time = None
    blank_b_absorbance = None
    
    print(f"Loading blank files...\n")
    
    blank_a_path = Path(blank_a_path)
    if blank_a_path.exists():
        print(f"✓ Found blank A file (214 nm): {blank_a_path.name}")
        blank_a_time, blank_a_absorbance = read_csv_file(blank_a_path)
        if blank_a_time is not None:
            print(f"  Data points: {len(blank_a_time)}\n")
    else:
        print(f"✗ Blank A file (214 nm) not found: {blank_a_path}\n")
    
    blank_b_path = Path(blank_b_path)
    if blank_b_path.exists():
        print(f"✓ Found blank B file (280 nm): {blank_b_path.name}")
        blank_b_time, blank_b_absorbance = read_csv_file(blank_b_path)
        if blank_b_time is not None:
            print(f"  Data points: {len(blank_b_time)}\n")
    else:
        print(f"✗ Blank B file (280 nm) not found: {blank_b_path}\n")
    
    return (blank_a_time, blank_a_absorbance), (blank_b_time, blank_b_absorbance)

# ============================================================================
# FUNCTION 3: Apply Background Correction
# ============================================================================

def apply_background_correction(absorbance, blank_time, blank_absorbance, sample_time):
    """Subtract blank absorbance from sample absorbance."""
    if blank_absorbance is None:
        print("  ⚠ No blank available, using uncorrected data")
        return absorbance
    
    if len(blank_time) != len(sample_time) or not np.allclose(blank_time, sample_time):
        blank_absorbance_interp = np.interp(sample_time, blank_time, blank_absorbance)
    else:
        blank_absorbance_interp = blank_absorbance
    
    corrected = absorbance - blank_absorbance_interp
    corrected = np.maximum(corrected, 0)
    
    return corrected

# ============================================================================
# FUNCTION 4: Gaussian Function for Peak Fitting
# ============================================================================

def gaussian(x, amp, mu, sigma):
    """Gaussian function for peak fitting."""
    return amp * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))

# ============================================================================
# FUNCTION 5: Detect Peaks
# ============================================================================

def detect_peaks_in_chromatogram(time, absorbance):
    """Detect major peaks (>10% of max intensity)."""
    max_abs = np.max(absorbance)
    height_threshold = max_abs * PEAK_HEIGHT_THRESHOLD
    
    try:
        peaks, properties = find_peaks(absorbance, height=height_threshold, distance=PEAK_DISTANCE)
        
        peak_data = []
        for peak_idx in peaks:
            peak_data.append({
                'time': time[peak_idx],
                'absorbance': absorbance[peak_idx],
                'index': peak_idx,
                'height': absorbance[peak_idx]
            })
        
        return sorted(peak_data, key=lambda x: x['time'])
    
    except Exception as e:
        print(f"  ⚠ Peak detection error: {e}")
        return []

# ============================================================================
# FUNCTION 6: Fit Single Peak with Gaussian
# ============================================================================

def fit_single_peak_gaussian(time, absorbance, peak_idx):
    """Fit a single peak with a Gaussian function."""
    try:
        left = max(0, peak_idx - 30)
        right = min(len(time), peak_idx + 30)
        
        time_fit = time[left:right]
        abs_fit = absorbance[left:right]
        
        amp_init = absorbance[peak_idx]
        mu_init = time[peak_idx]
        sigma_init = 0.08
        
        popt, _ = curve_fit(gaussian, time_fit, abs_fit,
                           p0=[amp_init, mu_init, sigma_init],
                           maxfev=5000)
        
        amp, mu, sigma = popt
        area = amp * sigma * np.sqrt(2 * np.pi)
        
        return {
            'amp': amp,
            'mu': mu,
            'sigma': sigma,
            'area': area,
            'success': True
        }
    
    except Exception as e:
        return {'success': False}

# ============================================================================
# FUNCTION 7: Detect and Fit Overlapping Peaks with TIC-Based 50% Threshold
# ============================================================================

def fit_overlapping_peaks(time, absorbance, peaks):
    """Detect and fit overlapping peaks with multi-Gaussian (only if total intensity > 50% of max peak)."""
    if len(peaks) < 2:
        return peaks
    
    try:
        # IMPORTANT: Find the MAJOR peak (maximum intensity in entire chromatogram)
        max_peak_intensity = max([p['absorbance'] for p in peaks])
        deconvolution_threshold_value = max_peak_intensity * DECONVOLUTION_THRESHOLD
        
        print(f"    Major peak intensity (TIC max): {max_peak_intensity:.2e}")
        print(f"    Deconvolution threshold (50% of major): {deconvolution_threshold_value:.2e}")
        
        # Detect overlapping peaks (closer than 0.3 min)
        overlapping_groups = []
        used = set()
        
        for i, peak1 in enumerate(peaks):
            if i in used:
                continue
            
            group = [i]
            used.add(i)
            
            for j, peak2 in enumerate(peaks[i+1:], i+1):
                if j not in used and abs(peak1['time'] - peak2['time']) < 0.3:
                    group.append(j)
                    used.add(j)
            
            if len(group) > 1:
                overlapping_groups.append(group)
        
        # Process overlapping groups - ONLY if TOTAL intensity > 50% of major peak
        for group in overlapping_groups:
            peak_times = [f"{peaks[i]['time']:.2f}" for i in group]
            
            # Calculate total intensity of overlapping peaks
            group_total_intensity = sum([peaks[i]['absorbance'] for i in group])
            
            # Check if total intensity exceeds 50% threshold
            deconv_needed = group_total_intensity > deconvolution_threshold_value
            
            if not deconv_needed:
                print(f"    ℹ Overlapping peaks at times: {', '.join(peak_times)} min")
                print(f"      Total intensity: {group_total_intensity:.2e} ≤ 50% threshold ({deconvolution_threshold_value:.2e}) - NO deconvolution")
                continue
            
            print(f"    ℹ Overlapping peaks at times: {', '.join(peak_times)} min")
            print(f"      Total intensity: {group_total_intensity:.2e} > 50% threshold ({deconvolution_threshold_value:.2e}) - DECONVOLUTING")
            
            try:
                indices = [peaks[i]['index'] for i in group]
                min_idx = min(indices)
                max_idx = max(indices)
                
                left = max(0, min_idx - 30)
                right = min(len(time), max_idx + 30)
                
                time_fit = time[left:right]
                abs_fit = absorbance[left:right]
                
                params0 = []
                for i in group:
                    params0.extend([
                        peaks[i]['absorbance'] * 0.8,
                        peaks[i]['time'],
                        0.08
                    ])
                
                def multi_gaussian(t, *params):
                    result = np.zeros_like(t)
                    for k in range(0, len(params), 3):
                        result += gaussian(t, params[k], params[k+1], params[k+2])
                    return result
                
                # FIXED: Removed p_tol parameter
                popt, _ = curve_fit(multi_gaussian, time_fit, abs_fit,
                                   p0=params0, maxfev=10000)
                
                for idx, group_idx in enumerate(group):
                    k = idx * 3
                    amp = popt[k]
                    mu = popt[k+1]
                    sigma = popt[k+2]
                    
                    area = amp * sigma * np.sqrt(2 * np.pi)
                    
                    peaks[group_idx]['area'] = area
                    peaks[group_idx]['is_fitted'] = True
                    peaks[group_idx]['fit_params'] = {
                        'amp': amp,
                        'mu': mu,
                        'sigma': sigma
                    }
                    print(f"      Peak fitted: μ={mu:.3f} min, σ={sigma:.4f}, Area={area:.2e}")
            
            except Exception as e:
                print(f"    ⚠ Multi-Gaussian fit failed: {e}")
        
        return peaks
    
    except Exception as e:
        print(f"  ⚠ Overlapping peak detection error: {e}")
        return peaks

# ============================================================================
# FUNCTION 8: Calculate Peak Areas with Gaussian Fitting
# ============================================================================

def calculate_peak_areas(time, absorbance, peaks):
    """Calculate peak areas by fitting all peaks as Gaussians."""
    if len(peaks) == 0:
        return []
    
    peak_areas = []
    
    for peak in peaks:
        peak_idx = peak['index']
        fit_result = fit_single_peak_gaussian(time, absorbance, peak_idx)
        
        if fit_result['success']:
            peak_areas.append({
                'time': peak['time'],
                'absorbance': peak['absorbance'],
                'area': fit_result['area'],
                'index': peak_idx,
                'is_fitted': True,
                'fit_params': {
                    'amp': fit_result['amp'],
                    'mu': fit_result['mu'],
                    'sigma': fit_result['sigma']
                }
            })
        else:
            left_bound = max(0, peak_idx - 20)
            right_bound = min(len(absorbance), peak_idx + 20)
            
            for j in range(peak_idx, max(0, peak_idx - 50), -1):
                if j == 0 or absorbance[j] < absorbance[j-1]:
                    left_bound = j
                    break
            
            for j in range(peak_idx, min(len(absorbance), peak_idx + 50)):
                if j == len(absorbance) - 1 or absorbance[j] < absorbance[j+1]:
                    right_bound = j
                    break
            
            time_slice = time[left_bound:right_bound]
            abs_slice = absorbance[left_bound:right_bound]
            area = trapezoid(abs_slice, time_slice)
            
            peak_areas.append({
                'time': peak['time'],
                'absorbance': peak['absorbance'],
                'area': area,
                'index': peak_idx,
                'is_fitted': False
            })
    
    # Fit overlapping peaks with multi-Gaussian (only if > 50% of major peak)
    peak_areas = fit_overlapping_peaks(time, absorbance, peak_areas)
    
    # Calculate percentages based on TOTAL AREA
    total_area = sum([p['area'] for p in peak_areas])
    
    if total_area > 0:
        for peak in peak_areas:
            peak['percentage'] = (peak['area'] / total_area) * 100
    else:
        for peak in peak_areas:
            peak['percentage'] = 0
    
    return peak_areas

# ============================================================================
# FUNCTION 9: Smart Annotation Positioning (FIXED - Handle List Correctly)
# ============================================================================

def calculate_smart_annotation_positions(peak_areas_list, max_abs, plot_height_fraction=0.35):
    """Calculate optimal annotation positions to avoid overlaps."""
    
    positions = {}
    
    # FIXED: peak_areas_list is now a list, not a dict
    sorted_peaks = sorted(enumerate(peak_areas_list), key=lambda x: x[1]['time'])
    
    # Define available heights for annotations (as fractions of max_abs)
    height_levels = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]  # Much more offset
    
    for idx, (orig_idx, peak) in enumerate(sorted_peaks):
        peak_height = peak['absorbance']
        
        # Find best non-overlapping height
        best_level = 0
        min_distance = float('inf')
        
        for level in height_levels:
            label_y = peak_height + (max_abs * level)
            
            # Check distance to other annotations
            min_dist_to_others = float('inf')
            for other_idx in positions:
                other_y = positions[other_idx]
                dist = abs(label_y - other_y)
                min_dist_to_others = min(min_dist_to_others, dist)
            
            # Prefer levels with maximum distance to others
            if min_dist_to_others > min_distance:
                min_distance = min_dist_to_others
                best_level = level
        
        label_y = peak_height + (max_abs * best_level)
        positions[orig_idx] = label_y
    
    return positions

# ============================================================================
# FUNCTION 10: Generate Full Plot with Smart Annotations (NO BORDERS)
# ============================================================================

def generate_plot(filepath, time, absorbance, wavelength):
    """Generate plot with Gaussian area shading and smart annotations."""
    
    output_filename = f"{filepath.stem}.png"
    output_path = Path(OUTPUT_DIRECTORY) / output_filename
    
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_NARROW)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    if ENABLE_PEAK_INTEGRATION:
        peaks = detect_peaks_in_chromatogram(time, absorbance)
        peak_areas = calculate_peak_areas(time, absorbance, peaks)
        
        if peak_areas:
            max_abs = np.max(absorbance)
            
            # FIRST: Shade all peak areas as Gaussians
            for i, peak in enumerate(peak_areas):
                if peak.get('fit_params'):
                    color_idx = i % len(COLORS)
                    fit_params = peak['fit_params']
                    
                    mu = fit_params['mu']
                    sigma = fit_params['sigma']
                    amp = fit_params['amp']
                    
                    x_gauss = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
                    y_gauss = gaussian(x_gauss, amp, mu, sigma)
                    
                    ax.fill_between(x_gauss, 0, y_gauss, alpha=0.3,
                                  color=COLORS[color_idx], zorder=3)
    
    # SECOND: Draw the main line
    ax.plot(time, absorbance, linewidth=2.5, color='black', zorder=10)
    
    # THIRD: Add smart annotations
    if ENABLE_PEAK_INTEGRATION and peak_areas:
        max_abs = np.max(absorbance)
        
        # FIXED: Pass list directly, not dict
        positions = calculate_smart_annotation_positions(peak_areas, max_abs)
        
        total_percentage = sum([p['percentage'] for p in peak_areas])
        print(f"  Peak Integration (Full Range):")
        
        for i, peak in enumerate(peak_areas):
            marker_color = '#2ca02c'
            
            ax.plot(peak['time'], peak['absorbance'], 'o',
                   color=marker_color, markersize=10, zorder=15)
            
            # Use smart positioning
            label_y = positions[i]
            label_text = f"{peak['percentage']:.1f}%\nArea: {peak['area']:.2e}"
            annotation_x_offset = 1.2  # minutes to the right (adjust this value)

            # FIXED: Removed edgecolor and linewidth for borderless boxes
            ax.text(peak['time'] + annotation_x_offset, label_y, label_text,
                   ha='center', va='bottom',
                   fontsize=9, fontweight='bold',
                   color=marker_color,
                   bbox=dict(boxstyle='round,pad=0.4',
                           facecolor='white',
                           edgecolor='grey',
                           alpha=0.9))
            
            fitted_status = "✓ GAUSSIAN FITTED" if peak.get('fit_params') else "⊘ TRAPEZOIDAL"
            print(f"    Peak #{i+1}: t={peak['time']:.2f} min | {peak['percentage']:.1f}% | Area={peak['area']:.2e} | {fitted_status}")
        
        print(f"    Total: {total_percentage:.1f}%")
    
    ax.set_xlabel('Time (min)', fontsize=26, fontweight='bold', labelpad=15)
    ax.set_ylabel(f'Absorbance {wavelength} nm', fontsize=26, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='major', width=2.5, length=10)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)
    
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved: {output_filename}")
    return output_path

# ============================================================================
# FUNCTION 11: Generate SHORT Plot (NO BORDERS)
# ============================================================================

def generate_plot_short(filepath, time, absorbance, wavelength):
    """Generate SHORT range plot with smart annotations."""
    
    output_filename = f"{filepath.stem}_SHORT.png"
    output_path = Path(OUTPUT_DIRECTORY) / output_filename
    
    mask = (time >= 3.0) & (time <= 15.0)
    filtered_time = time[mask]
    filtered_absorbance = absorbance[mask]
    
    if len(filtered_time) == 0:
        print(f"  ⚠ No data in 3-15 min range for SHORT plot")
        return None
    
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_EXTRA_NARROW)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    if ENABLE_PEAK_INTEGRATION:
        peaks = detect_peaks_in_chromatogram(filtered_time, filtered_absorbance)
        peak_areas = calculate_peak_areas(filtered_time, filtered_absorbance, peaks)
        
        if peak_areas:
            max_abs = np.max(filtered_absorbance)
            
            # FIRST: Shade areas
            for i, peak in enumerate(peak_areas):
                if peak.get('fit_params'):
                    color_idx = i % len(COLORS)
                    fit_params = peak['fit_params']
                    
                    mu = fit_params['mu']
                    sigma = fit_params['sigma']
                    amp = fit_params['amp']
                    
                    x_gauss = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
                    y_gauss = gaussian(x_gauss, amp, mu, sigma)
                    
                    ax.fill_between(x_gauss, 0, y_gauss, alpha=0.3,
                                  color=COLORS[color_idx], zorder=3)
    
    # SECOND: Draw line
    ax.plot(filtered_time, filtered_absorbance, linewidth=2.5, color='black', zorder=10)
    
    # THIRD: Smart annotations
    if ENABLE_PEAK_INTEGRATION and peak_areas:
        max_abs = np.max(filtered_absorbance)
        # FIXED: Pass list directly
        positions = calculate_smart_annotation_positions(peak_areas, max_abs)
        
        total_percentage = sum([p['percentage'] for p in peak_areas])
        print(f"  Peak Integration (3-15 min range):")
        
        for i, peak in enumerate(peak_areas):
            marker_color = '#2ca02c'
            
            ax.plot(peak['time'], peak['absorbance'], 'o',
                   color=marker_color, markersize=10, zorder=15)
            
            label_y = positions[i]
            label_text = f"{peak['percentage']:.1f}%\nArea: {peak['area']:.2e}"
            annotation_x_offset = 1.2  # minutes to the right (adjust this value)

            # FIXED: Removed edgecolor and linewidth for borderless boxes
            ax.text(peak['time'] + annotation_x_offset, label_y, label_text,
                   ha='center', va='bottom',
                   fontsize=9, fontweight='bold',
                   color=marker_color,
                   bbox=dict(boxstyle='round,pad=0.4',
                           facecolor='white',
                           edgecolor='grey',
                           alpha=0.9))
            
            fitted_status = "✓ GAUSSIAN FITTED" if peak.get('fit_params') else "⊘ TRAPEZOIDAL"
            print(f"    Peak #{i+1}: t={peak['time']:.2f} min | {peak['percentage']:.1f}% | Area={peak['area']:.2e} | {fitted_status}")
        
        print(f"    Total: {total_percentage:.1f}%")
    
    ax.set_xlabel('Time (min)', fontsize=26, fontweight='bold', labelpad=15)
    ax.set_ylabel(f'Absorbance {wavelength} nm', fontsize=26, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='major', width=2.5, length=10)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)
    
    ax.set_xlim(3.0, 15.0)
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved: {output_filename}")
    return output_path

# ============================================================================
# FUNCTION 12: Detect File Type
# ============================================================================

def get_file_type(filename):
    """Determine if file is A (214 nm) or B (280 nm)."""
    filename_upper = filename.upper()
    
    if '_DAD1A' in filename_upper or filename_upper.endswith('A.CSV'):
        return 'A', '214'
    elif '_DAD1B' in filename_upper or filename_upper.endswith('B.CSV'):
        return 'B', '280'
    else:
        return None, None

# ============================================================================
# FUNCTION 13: Process All Files
# ============================================================================

def process_all_csv_files():
    """Process all CSV files in the directory."""
    
    input_path = Path(INPUT_DIRECTORY)
    output_path = Path(OUTPUT_DIRECTORY)
    
    if not input_path.exists():
        print(f"✗ Directory not found: {INPUT_DIRECTORY}")
        return
    
    output_path.mkdir(exist_ok=True)
    
    (blank_a_time, blank_a_absorbance), (blank_b_time, blank_b_absorbance) = load_blank_files(
        BLANK_FILE_A_PATH,
        BLANK_FILE_B_PATH
    )
    
    all_csv_files = sorted(list(input_path.glob('*.CSV'))) + sorted(list(input_path.glob('*.csv')))
    all_csv_files = list(dict.fromkeys(all_csv_files))
    
    if not all_csv_files:
        print(f"✗ No CSV files found in: {INPUT_DIRECTORY}")
        return
    
    print(f"{'='*70}")
    print(f"Processing {len(all_csv_files)} CSV files")
    if ENABLE_PEAK_INTEGRATION:
        print(f"Peak Integration: ON (Deconvolution if > 50% of TIC major peak)")
    else:
        print(f"Peak Integration: OFF")
    print(f"{'='*70}\n")
    
    success_a = 0
    success_b = 0
    failed = 0
    skipped = 0
    
    for filepath in all_csv_files:
        try:
            if 'BLANK' in filepath.name.upper():
                print(f"⊘ Skipping blank file: {filepath.name}")
                skipped += 1
                continue
            
            file_letter, wavelength = get_file_type(filepath.name)
            
            if file_letter is None:
                print(f"⚠ Skipping {filepath.name} (cannot determine file type)")
                skipped += 1
                continue
            
            file_type = f"{file_letter} ({wavelength} nm)"
            print(f"Processing: {filepath.name} ({file_type})")
            
            time, absorbance = read_csv_file(filepath)
            
            if time is None or absorbance is None:
                failed += 1
                continue
            
            if file_letter == 'A':
                blank_time = blank_a_time
                blank_absorbance = blank_a_absorbance
                print(f"  ✓ Using blank A (214 nm) for background subtraction")
            else:
                blank_time = blank_b_time
                blank_absorbance = blank_b_absorbance
                print(f"  ✓ Using blank B (280 nm) for background subtraction")
            
            if blank_absorbance is not None:
                absorbance_corrected = apply_background_correction(
                    absorbance, blank_time, blank_absorbance, time
                )
                print(f"  Background corrected")
            else:
                absorbance_corrected = absorbance
                print(f"  No background correction applied")
            
            generate_plot(filepath, time, absorbance_corrected, wavelength)
            generate_plot_short(filepath, time, absorbance_corrected, wavelength)
            
            if file_letter == 'A':
                success_a += 1
            else:
                success_b += 1
        
        except Exception as e:
            print(f"✗ Error processing {filepath.name}: {e}")
            import traceback
            traceback.print_exc()
            failed += 1
    
    print(f"\n{'='*70}")
    print(f"Complete!")
    print(f"  ✓ Processed A files (214 nm): {success_a}")
    print(f"  ✓ Processed B files (280 nm): {success_b}")
    if skipped > 0:
        print(f"  ⊘ Skipped: {skipped}")
    if failed > 0:
        print(f"  ✗ Failed: {failed}")
    print(f"{'='*70}\n")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    process_all_csv_files()